# Tune Logistic Regression on Untouched and Retrained MPNet SentenceTransformer

This notebook loads both saved encoders, computes embeddings for the shared split, tunes One-vs-Rest Logistic Regression on validation macro F1, tunes per-aspect thresholds on validation predictions, and saves one classifier bundle per encoder.

Run `01-mpnet_retraining.ipynb` first.


In [ ]:
from pathlib import Path
import json
import os

import joblib
import numpy as np
import pandas as pd
from joblib import Parallel, delayed, parallel_config
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import ParameterGrid
from sklearn.multiclass import OneVsRestClassifier
from sentence_transformers import SentenceTransformer


In [ ]:
path = Path.cwd().resolve()
for _ in range(8):
    if (path / "data" / "Restaurant_ABSA_processed.csv").exists():
        PROJECT_ROOT = path
        break
    path = path.parent
else:
    raise FileNotFoundError("Could not locate the project root")

DATA_PATH = PROJECT_ROOT / "data" / "Restaurant_ABSA_processed.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "MPNet"
MODEL_DIR = PROJECT_ROOT / "models" / "mpnet"
SPLIT_PATH = OUTPUT_DIR / "data_split.npz"

ENCODER_PATHS = {
    "unretrained": MODEL_DIR / "unretrained_encoder",
    "retrained": MODEL_DIR / "retrained_encoder",
}

if not SPLIT_PATH.exists():
    raise FileNotFoundError("Run 01-mpnet_retraining.ipynb before this notebook")

for name, encoder_path in ENCODER_PATHS.items():
    if not encoder_path.exists():
        raise FileNotFoundError(f"Missing {name} encoder: {encoder_path}")


In [ ]:
ASPECT_COLS = ["food", "price", "service", "ambiance", "miscellaneous"]
TEXT_COL = "review_cleaned"

df = pd.read_csv(DATA_PATH).dropna(subset=[TEXT_COL]).reset_index(drop=True)
texts = df[TEXT_COL].astype(str).to_numpy()
labels = df[ASPECT_COLS].to_numpy(dtype=np.int8)

split = np.load(SPLIT_PATH)
train_idx = split["train_idx"]
val_idx = split["val_idx"]
test_idx = split["test_idx"]

X_train_text = texts[train_idx].tolist()
X_val_text = texts[val_idx].tolist()
y_train = labels[train_idx]
y_val = labels[val_idx]

print("Train:", len(train_idx))
print("Validation:", len(val_idx))
print("Held-out test:", len(test_idx))


In [ ]:
GRID = {
    "C": [0.01, 0.05, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0],
    "class_weight": [None, "balanced"],
    "solver": ["liblinear", "lbfgs"],
}

N_JOBS = min(4, os.cpu_count() or 1)
NORMALIZE_EMBEDDINGS = True
BATCH_SIZE = 32


In [ ]:
def fit_one(params, X_train, X_val, y_train, y_val):
    model = OneVsRestClassifier(
        LogisticRegression(
            C=params["C"],
            class_weight=params["class_weight"],
            solver=params["solver"],
            max_iter=3000,
            random_state=42,
        ),
        n_jobs=1,
    )
    model.fit(X_train, y_train)
    probabilities = model.predict_proba(X_val)
    predictions = (probabilities >= 0.5).astype(np.int8)
    per_class = f1_score(
        y_val,
        predictions,
        average=None,
        zero_division=0,
    )

    result = {
        **params,
        "macro_f1": f1_score(
            y_val,
            predictions,
            average="macro",
            zero_division=0,
        ),
        "micro_f1": f1_score(
            y_val,
            predictions,
            average="micro",
            zero_division=0,
        ),
    }
    for index, aspect in enumerate(ASPECT_COLS):
        result[f"{aspect}_f1"] = per_class[index]
    return result


def run_grid_search(X_train, X_val):
    parameter_grid = list(ParameterGrid(GRID))
    with parallel_config(
        backend="loky",
        n_jobs=N_JOBS,
        inner_max_num_threads=1,
    ):
        results = Parallel(verbose=10, batch_size=1)(
            delayed(fit_one)(
                params,
                X_train,
                X_val,
                y_train,
                y_val,
            )
            for params in parameter_grid
        )

    return (
        pd.DataFrame(results)
        .sort_values(["macro_f1", "micro_f1"], ascending=False)
        .reset_index(drop=True)
    )


In [ ]:
def tune_thresholds(y_true, probabilities):
    thresholds = {}
    threshold_scores = {}

    for index, aspect in enumerate(ASPECT_COLS):
        best_threshold = 0.5
        best_score = -1.0

        for threshold in np.arange(0.05, 0.951, 0.01):
            predictions = (
                probabilities[:, index] >= threshold
            ).astype(np.int8)
            score = f1_score(
                y_true[:, index],
                predictions,
                zero_division=0,
            )
            if score > best_score:
                best_score = score
                best_threshold = threshold

        thresholds[aspect] = round(float(best_threshold), 2)
        threshold_scores[aspect] = float(best_score)

    return thresholds, threshold_scores


def apply_thresholds(probabilities, thresholds):
    predictions = np.zeros_like(probabilities, dtype=np.int8)
    for index, aspect in enumerate(ASPECT_COLS):
        predictions[:, index] = (
            probabilities[:, index] >= thresholds[aspect]
        ).astype(np.int8)

    empty_rows = np.flatnonzero(predictions.sum(axis=1) == 0)
    for row in empty_rows:
        predictions[row, np.argmax(probabilities[row])] = 1
    return predictions


## Tune and save one classifier for each encoder

Only train and validation data are used here. The test split remains untouched for `03-mpnet_evaluate.ipynb`.


In [ ]:
all_grid_results = []
summary_rows = []

for encoder_name, encoder_path in ENCODER_PATHS.items():
    print(f"\n===== {encoder_name.upper()} =====")
    encoder = SentenceTransformer(str(encoder_path))

    X_train = encoder.encode(
        X_train_text,
        batch_size=BATCH_SIZE,
        normalize_embeddings=NORMALIZE_EMBEDDINGS,
        convert_to_numpy=True,
        show_progress_bar=True,
    )
    X_val = encoder.encode(
        X_val_text,
        batch_size=BATCH_SIZE,
        normalize_embeddings=NORMALIZE_EMBEDDINGS,
        convert_to_numpy=True,
        show_progress_bar=True,
    )

    results_df = run_grid_search(X_train, X_val)
    results_df.insert(0, "encoder", encoder_name)
    all_grid_results.append(results_df)

    best_row = results_df.iloc[0]
    best_params = {
        "C": float(best_row["C"]),
        "class_weight": (
            None
            if pd.isna(best_row["class_weight"])
            else best_row["class_weight"]
        ),
        "solver": str(best_row["solver"]),
    }

    classifier = OneVsRestClassifier(
        LogisticRegression(
            **best_params,
            max_iter=3000,
            random_state=42,
        ),
        n_jobs=1,
    )
    classifier.fit(X_train, y_train)
    val_probabilities = classifier.predict_proba(X_val)
    thresholds, threshold_scores = tune_thresholds(
        y_val,
        val_probabilities,
    )
    val_predictions = apply_thresholds(
        val_probabilities,
        thresholds,
    )

    classifier_dir = MODEL_DIR / "classifiers" / encoder_name
    classifier_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump(classifier, classifier_dir / "model.joblib")

    metadata = {
        "encoder_name": encoder_name,
        "encoder_path": str(encoder_path.relative_to(PROJECT_ROOT)),
        "normalize_embeddings": NORMALIZE_EMBEDDINGS,
        "batch_size": BATCH_SIZE,
        "aspect_cols": ASPECT_COLS,
        "text_col": TEXT_COL,
        "parameters": best_params,
        "thresholds": thresholds,
        "validation_threshold_f1": threshold_scores,
        "validation_macro_f1": float(
            f1_score(
                y_val,
                val_predictions,
                average="macro",
                zero_division=0,
            )
        ),
        "validation_micro_f1": float(
            f1_score(
                y_val,
                val_predictions,
                average="micro",
                zero_division=0,
            )
        ),
    }
    with (classifier_dir / "metadata.json").open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(metadata, file, indent=2)

    summary_rows.append({
        "encoder": encoder_name,
        "C": best_params["C"],
        "class_weight": best_params["class_weight"],
        "solver": best_params["solver"],
        "validation_macro_f1": metadata["validation_macro_f1"],
        "validation_micro_f1": metadata["validation_micro_f1"],
    })

grid_results_df = pd.concat(all_grid_results, ignore_index=True)
grid_results_df.to_csv(
    OUTPUT_DIR / "logistic_grid_results.csv",
    index=False,
)

summary_df = pd.DataFrame(summary_rows).sort_values(
    "validation_macro_f1",
    ascending=False,
)
summary_df.to_csv(
    OUTPUT_DIR / "classifier_validation_summary.csv",
    index=False,
)
summary_df
